# Phase 2: Weight of Evidence (WoE) Binning & Information Value (IV) Feature Selection
This notebook implements true credit scorecard feature engineering. We bin numerical and categorical variables, replace raw inputs with their WoE values, and calculate Information Value (IV) to select stable risk features.


In [ ]:
import pandas as pd
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning


## 1. Load Cleaned Splits
We load the cleaned splits (Train/Test and Out-of-Time).


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()
print('Train size:', train_df.shape)
print('OOT size:', oot_df.shape)


## 2. Fit WoE Mappings
We fit automated binning (Decision-Tree based for numeric and class groupings for categorical) on the training set.


In [ ]:
numeric_features = [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
]
categorical_features = ['home_ownership', 'purpose']

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, numeric_features, categorical_features)


## 3. Information Value (IV) Ranking & Selection
Rank characteristics based on their predictive power (IV) and filter out useless (<0.02) and suspicious (>0.50) features.


In [ ]:
iv_df = pd.read_csv('outputs/scorecards/iv_report.csv') if pd.io.common.file_exists('outputs/scorecards/iv_report.csv') else pd.DataFrame(woe_model.iv_report)
print('IV Feature Rankings:')
print(iv_df)


## 4. Inspecting WoE Lookup Tables
Let's look at the generated Weight of Evidence mapping rules for a key feature like `annual_inc`.


In [ ]:
woe_df = pd.read_csv('outputs/scorecards/woe_tables.csv') if pd.io.common.file_exists('outputs/scorecards/woe_tables.csv') else None
if woe_df is not None:
    print(woe_df[woe_df['variable'] == 'annual_inc'])


## 5. WoE Transformation
We transform the raw training features to WoE values for model input.


In [ ]:
train_woe = woe_model.transform(train_df)
print('Transformed WoE DataFrame Head:')
print(train_woe.head())
